# ====================================================
# 1. HAM CSI - USER CLASSIFICATION
# ====================================================

In [39]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y_user = np.load("../src/y_user_new_identity.npy")
y_gesture = np.load("../src/y_gesture_new_identity.npy")

print(X.shape)
print(y_user.shape)
print(y_gesture.shape)

(1500, 200, 64)
(1500,)
(1500,)


In [40]:
still_idx = np.where(
    (y_gesture == 0) |
    (y_gesture == 4)
)[0]

X_still = X[still_idx]
y_still = y_user[still_idx]

print(X_still.shape)
print(np.unique(y_still, return_counts=True))

(600, 200, 64)
(array([0, 1, 2, 3, 4], dtype=int32), array([ 75,  75,  75,  75, 300]))


In [41]:
np.random.seed(42)

balanced_idx = []

for cls in np.unique(y_still):

    cls_idx = np.where(y_still == cls)[0]

    if cls == 4:
        cls_idx = np.random.choice(
            cls_idx,
            size=75,
            replace=False
        )

    balanced_idx.extend(cls_idx)

balanced_idx = np.array(balanced_idx)

X_still = X_still[balanced_idx]
y_still = y_still[balanced_idx]

print(X_still.shape)
print(np.unique(y_still, return_counts=True))

(375, 200, 64)
(array([0, 1, 2, 3, 4], dtype=int32), array([75, 75, 75, 75, 75]))


In [42]:
X_still = np.clip(
    X_still,
    -150,
    150
)

print(X_still.shape)

(375, 200, 64)


In [43]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y_still, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_still,
    y_cat,
    test_size=0.2,
    stratify=y_still,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(300, 200, 64)
(75, 200, 64)


In [44]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(
    Conv1D(
        64,
        kernel_size=5,
        activation="relu",
        input_shape=(200,64)
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        128,
        kernel_size=3,
        activation="relu"
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(GlobalAveragePooling1D())

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))

model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_8 (Conv1D)               │ (None, 196, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 196, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_8 (MaxPooling1D)  │ (None, 98, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 96, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 96, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_9 (MaxPooling1D)  │ (None, 48, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_4      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,109 (277.77 KB)

 Trainable params: 70,725 (276.27 KB)

 Non-trainable params: 384 (1.50 KB)

In [45]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=[early_stop]
)

Epoch 1/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - accuracy: 0.2667 - loss: 1.6727 - val_accuracy: 0.1833 - val_loss: 8.6672
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3792 - loss: 1.5337 - val_accuracy: 0.1833 - val_loss: 6.0526
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.3458 - loss: 1.4733 - val_accuracy: 0.1833 - val_loss: 6.4980
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3833 - loss: 1.3678 - val_accuracy: 0.2333 - val_loss: 3.1074
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.4750 - loss: 1.2395 - val_accuracy: 0.2333 - val_loss: 2.5621
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.4792 - loss: 1.2329 - val_accuracy: 0.3667 - val_loss: 2.5565
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5000 - loss: 1.1955 - val_accuracy: 0.3000 - val_loss: 2.3446
Epoch 8/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5250 - loss: 1.1042 - val_accuracy: 0.

In [46]:
loss, acc = model.evaluate(
    X_test,
    y_test
)

print("Accuracy:", acc)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4133 - loss: 1.4954
Accuracy: 0.41333332657814026


In [47]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

y_pred = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print(confusion_matrix(y_true, y_pred))

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "S01",
            "S02",
            "S03",
            "S04",
            "empty"
        ]
    )
)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
[[ 1  6  5  3  0]
 [ 0 13  0  0  2]
 [ 0  2  7  3  3]
 [ 0  7  2  1  5]
 [ 0  2  1  3  9]]
              precision    recall  f1-score   support

         S01       1.00      0.07      0.12        15
         S02       0.43      0.87      0.58        15
         S03       0.47      0.47      0.47        15
         S04       0.10      0.07      0.08        15
       empty       0.47      0.60      0.53        15

    accuracy                           0.41        75
   macro avg       0.49      0.41      0.36        75
weighted avg       0.49      0.41      0.36        75



# ====================================================
# 2. DELTA CSI - USER CLASSIFICATION
# ====================================================

In [48]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y_user = np.load("../src/y_user_new_identity.npy")

print("X:", X.shape)
print("y_user:", y_user.shape)

print(np.unique(y_user, return_counts=True))

X: (1500, 200, 64)
y_user: (1500,)
(array([0, 1, 2, 3, 4], dtype=int32), array([300, 300, 300, 300, 300]))


In [49]:
X_delta = np.diff(X, axis=1)

X_delta = np.clip(
    X_delta,
    -50,
    50
)

print("X_delta:", X_delta.shape)

print("Min:", X_delta.min())
print("Max:", X_delta.max())
print("Mean:", X_delta.mean())
print("Std:", X_delta.std())

X_delta: (1500, 199, 64)
Min: -50.0
Max: 50.0
Mean: 0.00033342352
Std: 1.9748566


In [50]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y_user, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y_user,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1200, 199, 64)
(300, 199, 64)


In [51]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(
    Conv1D(
        64,
        kernel_size=5,
        activation="relu",
        input_shape=(199, 64)
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        128,
        kernel_size=3,
        activation="relu"
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(GlobalAveragePooling1D())

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))

model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_10 (Conv1D)              │ (None, 195, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 195, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_10 (MaxPooling1D) │ (None, 97, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 95, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 95, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_11 (MaxPooling1D) │ (None, 47, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_5      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,109 (277.77 KB)

 Trainable params: 70,725 (276.27 KB)

 Non-trainable params: 384 (1.50 KB)

In [52]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 0.2260 - loss: 1.7343 - val_accuracy: 0.2083 - val_loss: 1.7281
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.3302 - loss: 1.5668 - val_accuracy: 0.2333 - val_loss: 2.0174
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.3771 - loss: 1.4963 - val_accuracy: 0.2542 - val_loss: 1.8038
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.4094 - loss: 1.3988 - val_accuracy: 0.2625 - val_loss: 1.7906
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.4219 - loss: 1.4473 - val_accuracy: 0.4083 - val_loss: 1.4327
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.4365 - loss: 1.3338 - val_accuracy: 0.4000 - val_loss: 1.4489
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.4354 - loss: 1.4383 - val_accuracy: 0.3375 - val_loss: 1.6732
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.4531 - loss: 1.4359 - val_accuracy: 0.

In [53]:
loss, acc = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Delta CSI Test Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5033 - loss: 1.1376
Delta CSI Test Accuracy: 0.503333330154419


In [54]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

y_pred_prob = model.predict(X_test)

y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

print(confusion_matrix(y_true, y_pred))

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "S01",
            "S02",
            "S03",
            "S04",
            "empty"
        ]
    )
)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
[[39  8  3 10  0]
 [ 6  8 32  9  5]
 [ 6  4 33 16  1]
 [ 3  7 12 34  4]
 [ 0  6  8  9 37]]
              precision    recall  f1-score   support

         S01       0.72      0.65      0.68        60
         S02       0.24      0.13      0.17        60
         S03       0.38      0.55      0.45        60
         S04       0.44      0.57      0.49        60
       empty       0.79      0.62      0.69        60

    accuracy                           0.50       300
   macro avg       0.51      0.50      0.50       300
weighted avg       0.51      0.50      0.50       300



# ====================================================
# 3. DELTA CSI + RANDOM FOREST
# ====================================================

In [55]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

X_delta = np.diff(X, axis=1)
X_delta = np.clip(X_delta, -50, 50)

print(X_delta.shape)

(1500, 199, 64)


In [56]:
features = []

for sample in X_delta:

    mean_feat = np.mean(sample, axis=0)
    std_feat = np.std(sample, axis=0)

    min_feat = np.min(sample, axis=0)
    max_feat = np.max(sample, axis=0)

    energy_feat = np.sum(sample**2, axis=0)

    feature_vector = np.concatenate([
        mean_feat,
        std_feat,
        min_feat,
        max_feat,
        energy_feat
    ])

    features.append(feature_vector)

X_feat = np.array(features)

print(X_feat.shape)

(1500, 320)


In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_feat,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(1200, 320)
(300, 320)


In [58]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [59]:
acc = rf.score(X_test, y_test)

print("RF Accuracy:", acc)

RF Accuracy: 0.4766666666666667


# ====================================================
# 4. Delta CSI + Normalizasyon
# ====================================================

In [60]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

print(X.shape)
print(y.shape)

(1500, 200, 64)
(1500,)


In [61]:
X_delta = np.diff(X, axis=1)

mean = X_delta.mean(axis=(1,2), keepdims=True)
std = X_delta.std(axis=(1,2), keepdims=True)

X_delta = (X_delta - mean) / (std + 1e-8)

X_delta = np.clip(X_delta, -5, 5)

print(X_delta.shape)

print("Min:", X_delta.min())
print("Max:", X_delta.max())
print("Mean:", X_delta.mean())
print("Std:", X_delta.std())

(1500, 199, 64)
Min: -5.0
Max: 5.0
Mean: -1.2752776e-06
Std: 0.96295875


In [62]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1200, 199, 64)
(300, 199, 64)


In [63]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(
    Conv1D(
        64,
        kernel_size=5,
        activation="relu",
        input_shape=(199,64)
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        128,
        kernel_size=3,
        activation="relu"
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(GlobalAveragePooling1D())

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))

model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 195, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 195, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_12 (MaxPooling1D) │ (None, 97, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_13 (Conv1D)              │ (None, 95, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 95, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_13 (MaxPooling1D) │ (None, 47, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_6      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,109 (277.77 KB)

 Trainable params: 70,725 (276.27 KB)

 Non-trainable params: 384 (1.50 KB)

In [64]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.2510 - loss: 1.7182 - val_accuracy: 0.2458 - val_loss: 1.6219
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.3365 - loss: 1.5321 - val_accuracy: 0.2042 - val_loss: 1.6625
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.3885 - loss: 1.4466 - val_accuracy: 0.2083 - val_loss: 1.7088
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.4240 - loss: 1.3750 - val_accuracy: 0.3583 - val_loss: 1.4337
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.4333 - loss: 1.3440 - val_accuracy: 0.3583 - val_loss: 1.4980
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.4729 - loss: 1.2993 - val_accuracy: 0.3958 - val_loss: 1.4025
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.4833 - loss: 1.2726 - val_accuracy: 0.3750 - val_loss: 1.3940
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.4719 - loss: 1.2825 - val_accuracy: 0.

In [65]:
loss, acc = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Normalized Delta Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5067 - loss: 1.4043
Normalized Delta Accuracy: 0.5066666603088379


In [66]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

y_pred_prob = model.predict(X_test)

y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

print(confusion_matrix(y_true, y_pred))

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "S01",
            "S02",
            "S03",
            "S04",
            "empty"
        ]
    )
)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
[[40 10  0  7  3]
 [ 5 22 14  8 11]
 [11 19 15  8  7]
 [ 7  9  3 25 16]
 [ 0  6  0  4 50]]
              precision    recall  f1-score   support

         S01       0.63      0.67      0.65        60
         S02       0.33      0.37      0.35        60
         S03       0.47      0.25      0.33        60
         S04       0.48      0.42      0.45        60
       empty       0.57      0.83      0.68        60

    accuracy                           0.51       300
   macro avg       0.50      0.51      0.49       300
weighted avg       0.50      0.51      0.49       300



 # ====================================================
# 4.  Delta CSI + CNN-LSTM
# ====================================================

In [73]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

X_delta = np.diff(X, axis=1)
X_delta = np.clip(X_delta, -50, 50)

print(X_delta.shape)

(1500, 199, 64)


In [74]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1200, 199, 64)
(300, 199, 64)


In [75]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

model = Sequential()

# CNN kısmı
model.add(
    Conv1D(
        filters=64,
        kernel_size=5,
        activation="relu",
        input_shape=(199,64)
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        filters=128,
        kernel_size=3,
        activation="relu"
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

# LSTM kısmı
model.add(
    LSTM(
        64,
        return_sequences=False
    )
)

model.add(Dropout(0.4))

# Dense
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_16 (Conv1D)              │ (None, 195, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 195, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_16 (MaxPooling1D) │ (None, 97, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_17 (Conv1D)              │ (None, 95, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 95, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_17 (MaxPooling1D) │ (None, 47, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 99,909 (390.27 KB)

 Trainable params: 99,525 (388.77 KB)

 Non-trainable params: 384 (1.50 KB)

In [76]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 50ms/step - accuracy: 0.2375 - loss: 1.6508 - val_accuracy: 0.2208 - val_loss: 1.6037
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.3250 - loss: 1.5477 - val_accuracy: 0.3125 - val_loss: 1.5983
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.3406 - loss: 1.5014 - val_accuracy: 0.2625 - val_loss: 1.5696
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.4031 - loss: 1.3926 - val_accuracy: 0.3083 - val_loss: 1.5232
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.4323 - loss: 1.3560 - val_accuracy: 0.3125 - val_loss: 1.4728
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.4677 - loss: 1.2714 - val_accuracy: 0.3458 - val_loss: 1.5226
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.5115 - loss: 1.1742 - val_accuracy: 0.3167 - val_loss: 1.5581
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.5448 - loss: 1.1140 - val_accuracy: 0.

KeyboardInterrupt: 

In [ ]:
loss, acc = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Delta CNN-LSTM Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4100 - loss: 2.1395
Delta CNN-LSTM Accuracy: 0.4099999964237213


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_pred_prob = model.predict(X_test)

y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

print(confusion_matrix(y_true, y_pred))

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "S01",
            "S02",
            "S03",
            "S04",
            "empty"
        ]
    )
)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
[[30  9  9 10  2]
 [ 7 14 15 13 11]
 [ 5 18 17 14  6]
 [10  9 10 21 10]
 [ 3  2  4 10 41]]
              precision    recall  f1-score   support

         S01       0.55      0.50      0.52        60
         S02       0.27      0.23      0.25        60
         S03       0.31      0.28      0.30        60
         S04       0.31      0.35      0.33        60
       empty       0.59      0.68      0.63        60

    accuracy                           0.41       300
   macro avg       0.40      0.41      0.41       300
weighted avg       0.40      0.41      0.41       300



 # ====================================================
# 4.  Delta CSI + Gesture Classification
# ====================================================

In [77]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y_gesture = np.load("../src/y_gesture_new_identity.npy")

X_delta = np.diff(X, axis=1)
X_delta = np.clip(X_delta, -50, 50)

print("X_delta:", X_delta.shape)
print("Gesture dağılımı:", np.unique(y_gesture, return_counts=True))

X_delta: (1500, 199, 64)
Gesture dağılımı: (array([0, 1, 2, 3, 4], dtype=int32), array([300, 300, 300, 300, 300]))


In [78]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y_gesture, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y_gesture,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1200, 199, 64)
(300, 199, 64)


In [79]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(Conv1D(64, kernel_size=5, activation="relu", input_shape=(199,64)))
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(Conv1D(128, kernel_size=3, activation="relu"))
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(GlobalAveragePooling1D())

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))

model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_18 (Conv1D)              │ (None, 195, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 195, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_18 (MaxPooling1D) │ (None, 97, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_19 (Conv1D)              │ (None, 95, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 95, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_19 (MaxPooling1D) │ (None, 47, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_7      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,109 (277.77 KB)

 Trainable params: 70,725 (276.27 KB)

 Non-trainable params: 384 (1.50 KB)

In [80]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.2115 - loss: 1.8204 - val_accuracy: 0.2000 - val_loss: 2.1037
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.2698 - loss: 1.6814 - val_accuracy: 0.2000 - val_loss: 1.6588
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.3198 - loss: 1.6155 - val_accuracy: 0.2000 - val_loss: 1.8110
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.2927 - loss: 1.5951 - val_accuracy: 0.2167 - val_loss: 1.6725
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.3354 - loss: 1.5598 - val_accuracy: 0.2750 - val_loss: 1.5905
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.3229 - loss: 1.5547 - val_accuracy: 0.2542 - val_loss: 1.6919
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.3323 - loss: 1.5959 - val_accuracy: 0.3333 - val_loss: 1.5342
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.3604 - loss: 1.5487 - val_accuracy: 0.

In [81]:
loss, acc = model.evaluate(X_test, y_test, verbose=1)

print("Delta Gesture Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3133 - loss: 1.9721
Delta Gesture Accuracy: 0.31333333253860474


In [82]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

gesture_names = ["still", "hand_clap", "horizontal_arm_wave", "bend", "empty"]

y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=gesture_names))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
[[14 23 20  0  3]
 [24 24 10  0  2]
 [12 23 22  0  3]
 [15 31 12  0  2]
 [ 4  9 13  0 34]]
                     precision    recall  f1-score   support

              still       0.20      0.23      0.22        60
          hand_clap       0.22      0.40      0.28        60
horizontal_arm_wave       0.29      0.37      0.32        60
               bend       0.00      0.00      0.00        60
              empty       0.77      0.57      0.65        60

           accuracy                           0.31       300
          macro avg       0.30      0.31      0.29       300
       weighted avg       0.30      0.31      0.29       300



/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

################# improved delta cnn user

In [7]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

X_delta = np.diff(X, axis=1)
X_delta = np.clip(X_delta, -50, 50)

print(X_delta.shape)

(1500, 199, 64)


In [8]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(
    Conv1D(
        64,
        kernel_size=7,
        activation="relu",
        input_shape=(199,64)
    )
)
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        128,
        kernel_size=5,
        activation="relu"
    )
)
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        256,
        kernel_size=3,
        activation="relu"
    )
)
model.add(BatchNormalization())

model.add(GlobalAveragePooling1D())

model.add(Dense(256, activation="relu"))
model.add(Dropout(0.5))

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_6 (Conv1D)               │ (None, 193, 64)        │        28,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 193, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 96, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 92, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 92, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 46, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 44, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 44, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 269,509 (1.03 MB)

 Trainable params: 268,613 (1.02 MB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=12,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - accuracy: 0.2583 - loss: 1.6348 - val_accuracy: 0.2042 - val_loss: 1.7240
Epoch 2/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.3969 - loss: 1.4378 - val_accuracy: 0.2792 - val_loss: 1.6225
Epoch 3/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.4198 - loss: 1.3754 - val_accuracy: 0.4083 - val_loss: 1.4075
Epoch 4/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.4333 - loss: 1.3542 - val_accuracy: 0.3625 - val_loss: 1.4387
Epoch 5/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.4760 - loss: 1.3447 - val_accuracy: 0.4167 - val_loss: 1.3998
Epoch 6/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.4510 - loss: 1.4393 - val_accuracy: 0.3917 - val_loss: 1.4318
Epoch 7/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.4594 - loss: 1.4541 - val_accuracy: 0.3583 - val_loss: 1.4915
Epoch 8/100
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.4719 - loss: 1.5147 - val_accuracy: 0.

In [12]:
loss, acc = model.evaluate(X_test, y_test)

print("Improved Delta CNN Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.4967 - loss: 3.3541
Improved Delta CNN Accuracy: 0.49666666984558105


In [13]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_pred_prob = model.predict(X_test)

y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

class_names = ["S01", "S02", "S03", "S04", "empty"]

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
[[53  2  2  3  0]
 [21  9 16  5  9]
 [18 13 24  4  1]
 [20  2 12 22  4]
 [ 6  3  1  9 41]]
              precision    recall  f1-score   support

         S01       0.45      0.88      0.60        60
         S02       0.31      0.15      0.20        60
         S03       0.44      0.40      0.42        60
         S04       0.51      0.37      0.43        60
       empty       0.75      0.68      0.71        60

    accuracy                           0.50       300
   macro avg       0.49      0.50      0.47       300
weighted avg       0.49      0.50      0.47       300



In [89]:
model.save("../models/IMPROVED_DELTA_CNN_USER_56.h5")

 # ====================================================
# Delta CSI + Improved CNN + Augmentation
# ====================================================

In [14]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

X_delta = np.diff(X, axis=1)
X_delta = np.clip(X_delta, -50, 50)

print(X_delta.shape)

(1500, 199, 64)


In [15]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)

(1200, 199, 64)


In [16]:
noise = np.random.normal(
    0,
    1.0,
    X_train.shape
)

X_noise = X_train + noise

print(X_noise.shape)

(1200, 199, 64)


In [17]:
X_shift = np.copy(X_train)

for i in range(len(X_shift)):
    shift = np.random.randint(-5, 6)
    X_shift[i] = np.roll(
        X_shift[i],
        shift,
        axis=0
    )

print(X_shift.shape)

(1200, 199, 64)


In [18]:
X_train_aug = np.concatenate([
    X_train,
    X_noise,
    X_shift
])

y_train_aug = np.concatenate([
    y_train,
    y_train,
    y_train
])

print(X_train_aug.shape)
print(y_train_aug.shape)

(3600, 199, 64)
(3600, 5)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(
    Conv1D(
        64,
        kernel_size=7,
        activation="relu",
        input_shape=(199,64)
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        128,
        kernel_size=5,
        activation="relu"
    )
)

model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        256,
        kernel_size=3,
        activation="relu"
    )
)

model.add(BatchNormalization())

model.add(GlobalAveragePooling1D())

model.add(Dense(256, activation="relu"))
model.add(Dropout(0.5))

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=12,
    restore_best_weights=True
)

history = model.fit(
    X_train_aug,
    y_train_aug,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
180/180 ━━━━━━━━━━━━━━━━━━━━ 8s 32ms/step - accuracy: 0.3139 - loss: 1.5600 - val_accuracy: 0.4667 - val_loss: 1.3573
Epoch 2/100
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.3979 - loss: 1.4203 - val_accuracy: 0.4528 - val_loss: 1.3206
Epoch 3/100
180/180 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.4267 - loss: 1.5825 - val_accuracy: 0.5153 - val_loss: 1.1434
Epoch 4/100
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - accuracy: 0.4045 - loss: 2.0674 - val_accuracy: 0.4611 - val_loss: 1.5246
Epoch 5/100
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.3931 - loss: 2.6283 - val_accuracy: 0.4708 - val_loss: 1.3807
Epoch 6/100
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.4007 - loss: 3.4274 - val_accuracy: 0.4806 - val_loss: 2.1123
Epoch 7/100
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.4007 - loss: 4.2798 - val_accuracy: 0.4847 - val_loss: 3.2555
Epoch 8/100
180/180 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.4187 - loss: 4.6715 - 

In [21]:
loss, acc = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Augmented Delta CNN Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5067 - loss: 1.4904
Augmented Delta CNN Accuracy: 0.5066666603088379


CNN + Attention

In [24]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

X_delta = np.diff(X, axis=1)
X_delta = np.clip(X_delta, -50, 50)

print("X_delta:", X_delta.shape)
print("y:", y.shape)
print(np.unique(y, return_counts=True))

X_delta: (1500, 199, 64)
y: (1500,)
(array([0, 1, 2, 3, 4], dtype=int32), array([300, 300, 300, 300, 300]))


In [25]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)

(1200, 199, 64) (300, 199, 64)


In [26]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, BatchNormalization,
    Dense, Dropout, GlobalAveragePooling1D,
    MultiHeadAttention, LayerNormalization, Add
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

inputs = Input(shape=(199, 64))

x = Conv1D(64, kernel_size=7, activation="relu", padding="same")(inputs)
x = BatchNormalization()(x)
x = MaxPooling1D(pool_size=2)(x)

x = Conv1D(128, kernel_size=5, activation="relu", padding="same")(x)
x = BatchNormalization()(x)
x = MaxPooling1D(pool_size=2)(x)

x = Conv1D(256, kernel_size=3, activation="relu", padding="same")(x)
x = BatchNormalization()(x)

# Attention block
attention_output = MultiHeadAttention(
    num_heads=4,
    key_dim=64
)(x, x)

x = Add()([x, attention_output])
x = LayerNormalization()(x)

x = GlobalAveragePooling1D()(x)

x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)

x = Dense(128, activation="relu")(x)
x = Dropout(0.4)(x)

outputs = Dense(5, activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_56"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 199, 64)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_12 (Conv1D)  │ (None, 199, 64)   │     28,736 │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 199, 64)   │        256 │ conv1d_12[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_8     │ (None, 99, 64)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 99, 128)   │     41,088 │ max_pooling1d_8[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 99, 128)   │        512 │ conv1d_13[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_9     │ (None, 49, 128)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 49, 256)   │     98,560 │ max_pooling1d_9[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 49, 256)   │      1,024 │ conv1d_14[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 49, 256)   │    263,168 │ batch_normalizat… │
│ (MultiHeadAttentio… │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 49, 256)   │          0 │ batch_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 49, 256)   │        512 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 256)       │     65,792 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 256)       │          0 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 128)       │     32,896 │ dropout_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 128)       │          0 │ dense_13[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 5)         │        645 │ dropout_10[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 533,189 (2.03 MB)

 Trainable params: 532,293 (2.03 MB)

 Non-trainable params: 896 (3.50 KB)

In [27]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=12,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 9s 85ms/step - accuracy: 0.3031 - loss: 1.5740 - val_accuracy: 0.2583 - val_loss: 1.7200
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 0.4000 - loss: 1.3909 - val_accuracy: 0.2625 - val_loss: 2.3497
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - accuracy: 0.4552 - loss: 1.3104 - val_accuracy: 0.2625 - val_loss: 2.0483
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.4979 - loss: 1.2861 - val_accuracy: 0.3667 - val_loss: 1.5921
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.4854 - loss: 1.3894 - val_accuracy: 0.3583 - val_loss: 1.5386
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.4875 - loss: 1.4171 - val_accuracy: 0.2792 - val_loss: 2.0567
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 59ms/step - accuracy: 0.4854 - loss: 1.5965 - val_accuracy: 0.3542 - val_loss: 1.6977
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.4740 - loss: 1.8149 - val_accuracy: 0.

In [28]:
loss, acc = model.evaluate(X_test, y_test, verbose=1)

print("Delta CNN Attention Accuracy:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.4100 - loss: 2.0598
Delta CNN Attention Accuracy: 0.4099999964237213


Session1+2 train, Session3 test

In [37]:
X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

print(X.shape)
print(y.shape)

(1500, 200, 64)
(1500,)


In [41]:
train_mask = session_labels != 3
test_mask = session_labels == 3

X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (1000, 200, 64)
Test : (500, 200, 64)


In [42]:
X_train = np.diff(X_train, axis=1)
X_test = np.diff(X_test, axis=1)

X_train = np.clip(X_train, -50, 50)
X_test = np.clip(X_test, -50, 50)

print(X_train.shape)
print(X_test.shape)

(1000, 199, 64)
(500, 199, 64)


In [43]:
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(y_train, 5)
y_test_cat = to_categorical(y_test, 5)

In [44]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, BatchNormalization,
    Dense, Dropout, GlobalAveragePooling1D,
    MultiHeadAttention, LayerNormalization, Add
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

inputs = Input(shape=(199, 64))

x = Conv1D(64, kernel_size=7, activation="relu", padding="same")(inputs)
x = BatchNormalization()(x)
x = MaxPooling1D(pool_size=2)(x)

x = Conv1D(128, kernel_size=5, activation="relu", padding="same")(x)
x = BatchNormalization()(x)
x = MaxPooling1D(pool_size=2)(x)

x = Conv1D(256, kernel_size=3, activation="relu", padding="same")(x)
x = BatchNormalization()(x)

# Attention block
attention_output = MultiHeadAttention(
    num_heads=4,
    key_dim=64
)(x, x)

x = Add()([x, attention_output])
x = LayerNormalization()(x)

x = GlobalAveragePooling1D()(x)

x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)

x = Dense(128, activation="relu")(x)
x = Dropout(0.4)(x)

outputs = Dense(5, activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_71"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 199, 64)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_15 (Conv1D)  │ (None, 199, 64)   │     28,736 │ input_layer_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 199, 64)   │        256 │ conv1d_15[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_10    │ (None, 99, 64)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_16 (Conv1D)  │ (None, 99, 128)   │     41,088 │ max_pooling1d_10… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 99, 128)   │        512 │ conv1d_16[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_11    │ (None, 49, 128)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_17 (Conv1D)  │ (None, 49, 256)   │     98,560 │ max_pooling1d_11… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 49, 256)   │      1,024 │ conv1d_17[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 49, 256)   │    263,168 │ batch_normalizat… │
│ (MultiHeadAttentio… │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 49, 256)   │          0 │ batch_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 49, 256)   │        512 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 256)       │     65,792 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 256)       │          0 │ dense_18[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 128)       │     32,896 │ dropout_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 128)       │          0 │ dense_19[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 5)         │        645 │ dropout_15[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 533,189 (2.03 MB)

 Trainable params: 532,293 (2.03 MB)

 Non-trainable params: 896 (3.50 KB)

In [45]:
history = model.fit(
    X_train,
    y_train_cat,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 8s 189ms/step - accuracy: 0.2275 - loss: 1.6419 - val_accuracy: 0.2350 - val_loss: 2.6295
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step - accuracy: 0.3550 - loss: 1.4668 - val_accuracy: 0.2500 - val_loss: 2.0395
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - accuracy: 0.4175 - loss: 1.3739 - val_accuracy: 0.2550 - val_loss: 2.5450
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.4387 - loss: 1.3402 - val_accuracy: 0.2900 - val_loss: 1.9104
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - accuracy: 0.4787 - loss: 1.3235 - val_accuracy: 0.3450 - val_loss: 1.5189
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - accuracy: 0.5038 - loss: 1.2817 - val_accuracy: 0.3350 - val_loss: 1.7011
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.4812 - loss: 1.3771 - val_accuracy: 0.3700 - val_loss: 1.7595
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.5038 - loss: 1.6588 - val_accuracy: 0

In [46]:
loss, acc = model.evaluate(
    X_test,
    y_test_cat,
    verbose=1
)

print("SESSION-BASED ACCURACY:", acc)

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.2380 - loss: 2.5177
SESSION-BASED ACCURACY: 0.23800000548362732


model çöktü ya sessionda neden train test ayırmak mantıklı gelmişiti bana farklı zamanlarda daha iyi olur diye düşünürken model çöktü
random split= %55.7
session split = %23.8

In [47]:
import numpy as np

y_pred_prob = model.predict(X_test)

y_pred = np.argmax(y_pred_prob, axis=1)

print(y_pred[:20])

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step
[1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [48]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[ 14 105   0   0   0]
 [  2 105   0   0   0]
 [  2  82   0   0   0]
 [  6  97   0   0   0]
 [  1  86   0   0   0]]


In [49]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "S01",
            "S02",
            "S03",
            "S04",
            "empty"
        ]
    )
)

              precision    recall  f1-score   support

         S01       0.56      0.12      0.19       119
         S02       0.22      0.98      0.36       107
         S03       0.00      0.00      0.00        84
         S04       0.00      0.00      0.00       103
       empty       0.00      0.00      0.00        87

    accuracy                           0.24       500
   macro avg       0.16      0.22      0.11       500
weighted avg       0.18      0.24      0.12       500



/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

In [50]:
for i,name in enumerate([
    "S01",
    "S02",
    "S03",
    "S04",
    "empty"
]):

    idx = (y_test == i)

    correct = np.sum(
        y_pred[idx] == y_test[idx]
    )

    total = np.sum(idx)

    print(
        f"{name}: {correct}/{total} = {100*correct/total:.1f}%"
    )

S01: 14/119 = 11.8%
S02: 105/107 = 98.1%
S03: 0/84 = 0.0%
S04: 0/103 = 0.0%
empty: 0/87 = 0.0%


*********************

In [51]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

X_delta = np.diff(X, axis=1)
X_delta = np.clip(X_delta, -50, 50)

features = []

for sample in X_delta:

    mean_feat = np.mean(sample, axis=0)
    std_feat = np.std(sample, axis=0)

    feature_vector = np.concatenate([
        mean_feat,
        std_feat
    ])

    features.append(feature_vector)

X_feat = np.array(features)

print(X_feat.shape)

(1500, 128)


In [52]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X_train, X_test, y_train, y_test = train_test_split(
    X_feat,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

svm = SVC(
    kernel="rbf",
    C=10,
    gamma="scale"
)

svm.fit(X_train, y_train)

acc = svm.score(X_test, y_test)

print("SVM Accuracy:", acc)

SVM Accuracy: 0.38666666666666666


****timesteps=100

In [53]:
import numpy as np

X = np.load("../src/X_identity_100.npy")
y = np.load("../src/y_user_identity_100.npy")

print(X.shape)
print(y.shape)

(1500, 100, 64)
(1500,)


In [54]:
X_delta = np.diff(X, axis=1)

X_delta = np.clip(
    X_delta,
    -50,
    50
)

print(X_delta.shape)

(1500, 99, 64)


In [55]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, 5)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1200, 99, 64)
(300, 99, 64)


In [56]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    BatchNormalization,
    MaxPooling1D,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(
    Conv1D(
        64,
        kernel_size=7,
        activation="relu",
        padding="same",
        input_shape=(99,64)
    )
)

model.add(BatchNormalization())

model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        128,
        kernel_size=5,
        activation="relu",
        padding="same"
    )
)

model.add(BatchNormalization())

model.add(MaxPooling1D(2))

model.add(
    Conv1D(
        256,
        kernel_size=3,
        activation="relu",
        padding="same"
    )
)

model.add(BatchNormalization())

model.add(GlobalAveragePooling1D())

model.add(Dense(256, activation="relu"))

model.add(Dropout(0.5))

model.add(Dense(128, activation="relu"))

model.add(Dropout(0.4))

model.add(Dense(5, activation="softmax"))

model.compile(
    optimizer=Adam(0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_18 (Conv1D)              │ (None, 99, 64)         │        28,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (None, 99, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_12 (MaxPooling1D) │ (None, 49, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_19 (Conv1D)              │ (None, 49, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_22          │ (None, 49, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_13 (MaxPooling1D) │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_20 (Conv1D)              │ (None, 24, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_23          │ (None, 24, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_6      │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 269,509 (1.03 MB)

 Trainable params: 268,613 (1.02 MB)

 Non-trainable params: 896 (3.50 KB)

In [57]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=12,
    restore_best_weights=True
)

In [58]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 58ms/step - accuracy: 0.2417 - loss: 1.6853 - val_accuracy: 0.2000 - val_loss: 7.5886
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.3333 - loss: 1.5053 - val_accuracy: 0.2167 - val_loss: 2.6075
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.4292 - loss: 1.3753 - val_accuracy: 0.2042 - val_loss: 3.4065
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.4635 - loss: 1.2742 - val_accuracy: 0.1917 - val_loss: 2.1804
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.5427 - loss: 1.1178 - val_accuracy: 0.2833 - val_loss: 1.8615
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.6302 - loss: 1.0034 - val_accuracy: 0.2250 - val_loss: 2.1794
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.6740 - loss: 0.8386 - val_accuracy: 0.3458 - val_loss: 1.5193
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.7510 - loss: 0.6763 - val_accuracy: 0.

In [59]:
loss, acc = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("100 TIMESTEP DELTA CNN ACCURACY:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.2800 - loss: 2.8778
100 TIMESTEP DELTA CNN ACCURACY: 0.2800000011920929


In [60]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

y_pred_prob = model.predict(X_test)

y_pred = np.argmax(
    y_pred_prob,
    axis=1
)

y_true = np.argmax(
    y_test,
    axis=1
)

print(
    confusion_matrix(
        y_true,
        y_pred
    )
)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "S01",
            "S02",
            "S03",
            "S04",
            "empty"
        ]
    )
)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
[[21  8  3 28  0]
 [ 3 25  3 29  0]
 [ 0 24  2 34  0]
 [ 4 16  3 36  1]
 [ 0 17  6 37  0]]
              precision    recall  f1-score   support

         S01       0.75      0.35      0.48        60
         S02       0.28      0.42      0.33        60
         S03       0.12      0.03      0.05        60
         S04       0.22      0.60      0.32        60
       empty       0.00      0.00      0.00        60

    accuracy                           0.28       300
   macro avg       0.27      0.28      0.24       300
weighted avg       0.27      0.28      0.24       300



eski best_user_82 nin modeli

In [61]:
import numpy as np

X = np.load("../src/X_identity_100.npy")
y = np.load("../src/y_user_identity_100.npy")

print(X.shape)
print(y.shape)

(1500, 100, 64)
(1500,)


In [62]:
X_normalized = []

for sample in X:

    sample_mean = np.mean(sample)
    sample_std = np.std(sample)

    sample = (
        sample - sample_mean
    ) / (sample_std + 1e-8)

    X_normalized.append(sample)

X = np.array(X_normalized)

print("Mean:", np.mean(X))
print("Std :", np.std(X))
print("Min :", np.min(X))
print("Max :", np.max(X))

Mean: -1.6291936e-10
Std : 1.0000002
Min : -43.589077
Max : 56.405926


In [63]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(1200, 100, 64)
(300, 100, 64)


In [64]:
noise = np.random.normal(
    0,
    0.02,
    X_train.shape
)

X_noise = X_train + noise

In [65]:
def temporal_shift(sample, shift=5):
    return np.roll(sample, shift, axis=0)

X_shifted = np.array([
    temporal_shift(
        x,
        shift=np.random.randint(-5,5)
    )
    for x in X_train
])

In [66]:
X_train_final = np.concatenate([
    X_train,
    X_noise,
    X_shifted
], axis=0)

y_train_final = np.concatenate([
    y_train,
    y_train,
    y_train
], axis=0)

print(X_train_final.shape)
print(y_train_final.shape)

(3600, 100, 64)
(3600,)


In [67]:
from tensorflow.keras.utils import to_categorical

num_classes = 5

y_train_cat = to_categorical(
    y_train_final,
    num_classes
)

y_test_cat = to_categorical(
    y_test,
    num_classes
)

In [68]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dense,
    Dropout
)

model = Sequential([

    Conv1D(
        32,
        kernel_size=3,
        activation='relu',
        input_shape=(100,64)
    ),

    MaxPooling1D(pool_size=2),

    Conv1D(
        64,
        kernel_size=3,
        activation='relu'
    ),

    MaxPooling1D(pool_size=2),

    LSTM(64),

    Dropout(0.5),

    Dense(
        64,
        activation='relu'
    ),

    Dense(
        num_classes,
        activation='softmax'
    )
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_21 (Conv1D)              │ (None, 98, 32)         │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_14 (MaxPooling1D) │ (None, 49, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_22 (Conv1D)              │ (None, 47, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_15 (MaxPooling1D) │ (None, 23, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,893 (194.89 KB)

 Trainable params: 49,893 (194.89 KB)

 Non-trainable params: 0 (0.00 B)

In [69]:
history = model.fit(
    X_train_final,
    y_train_cat,
    epochs=25,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/25
90/90 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - accuracy: 0.2101 - loss: 1.6556 - val_accuracy: 0.2167 - val_loss: 1.6040
Epoch 2/25
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.2160 - loss: 1.6135 - val_accuracy: 0.2861 - val_loss: 1.5663
Epoch 3/25
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.2896 - loss: 1.5378 - val_accuracy: 0.3028 - val_loss: 1.4978
Epoch 4/25
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.3858 - loss: 1.4067 - val_accuracy: 0.4306 - val_loss: 1.2973
Epoch 5/25
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.4201 - loss: 1.3073 - val_accuracy: 0.3819 - val_loss: 1.3377
Epoch 6/25
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.4604 - loss: 1.2309 - val_accuracy: 0.4361 - val_loss: 1.2488
Epoch 7/25
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.4826 - loss: 1.1818 - val_accuracy: 0.4944 - val_loss: 1.1368
Epoch 8/25
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.5208 - loss: 1.1140 - val_accuracy: 0.4889 - v

In [70]:
loss, acc = model.evaluate(
    X_test,
    y_test_cat
)

print("BEST_82 STYLE ACCURACY:", acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.4700 - loss: 2.0835
BEST_82 STYLE ACCURACY: 0.4699999988079071


In [71]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

predictions = model.predict(X_test)

pred_classes = np.argmax(
    predictions,
    axis=1
)

print(
    confusion_matrix(
        y_test,
        pred_classes
    )
)

print(
    classification_report(
        y_test,
        pred_classes,
        target_names=[
            "S01",
            "S02",
            "S03",
            "S04",
            "empty"
        ]
    )
)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
[[39  7  4 10  0]
 [ 9 20 17 13  1]
 [ 6 14 25 15  0]
 [13  5 14 24  4]
 [ 5  3  6 13 33]]
              precision    recall  f1-score   support

         S01       0.54      0.65      0.59        60
         S02       0.41      0.33      0.37        60
         S03       0.38      0.42      0.40        60
         S04       0.32      0.40      0.36        60
       empty       0.87      0.55      0.67        60

    accuracy                           0.47       300
   macro avg       0.50      0.47      0.48       300
weighted avg       0.50      0.47      0.48       300



----------------------------denemek için

In [72]:
import numpy as np

X = np.load("../src/X_new_identity.npy")
y = np.load("../src/y_user_new_identity.npy")

print(X.shape)
print(np.unique(y, return_counts=True))

(1500, 200, 64)
(array([0, 1, 2, 3, 4], dtype=int32), array([300, 300, 300, 300, 300]))


In [73]:
mask = y != 4

X_4user = X[mask]
y_4user = y[mask]

print(X_4user.shape)
print(np.unique(y_4user, return_counts=True))

(1200, 200, 64)
(array([0, 1, 2, 3], dtype=int32), array([300, 300, 300, 300]))


In [74]:
X_delta = np.diff(X_4user, axis=1)
X_delta = np.clip(X_delta, -50, 50)

print(X_delta.shape)
print(np.unique(y_4user, return_counts=True))

(1200, 199, 64)
(array([0, 1, 2, 3], dtype=int32), array([300, 300, 300, 300]))


In [75]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y_4user, 4)

X_train, X_test, y_train, y_test = train_test_split(
    X_delta,
    y_cat,
    test_size=0.2,
    stratify=y_4user,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(960, 199, 64)
(240, 199, 64)


In [76]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, MaxPooling1D
from tensorflow.keras.layers import GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(Conv1D(64, kernel_size=7, activation="relu", input_shape=(199,64)))
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(Conv1D(128, kernel_size=5, activation="relu"))
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(Conv1D(256, kernel_size=3, activation="relu"))
model.add(BatchNormalization())

model.add(GlobalAveragePooling1D())

model.add(Dense(256, activation="relu"))
model.add(Dropout(0.5))

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))

model.add(Dense(4, activation="softmax"))

model.compile(
    optimizer=Adam(0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_23 (Conv1D)              │ (None, 193, 64)        │        28,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_24          │ (None, 193, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_16 (MaxPooling1D) │ (None, 96, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_24 (Conv1D)              │ (None, 92, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_25          │ (None, 92, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_17 (MaxPooling1D) │ (None, 46, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_25 (Conv1D)              │ (None, 44, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_26          │ (None, 44, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_7      │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 269,380 (1.03 MB)

 Trainable params: 268,484 (1.02 MB)

 Non-trainable params: 896 (3.50 KB)

In [77]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=12,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 62ms/step - accuracy: 0.2852 - loss: 1.4261 - val_accuracy: 0.2708 - val_loss: 1.4180
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.3503 - loss: 1.3396 - val_accuracy: 0.3438 - val_loss: 1.3606
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.4193 - loss: 1.2623 - val_accuracy: 0.3281 - val_loss: 1.3219
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.4297 - loss: 1.2045 - val_accuracy: 0.3594 - val_loss: 1.3285
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.4961 - loss: 1.2082 - val_accuracy: 0.3594 - val_loss: 1.3252
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.5156 - loss: 1.1142 - val_accuracy: 0.3906 - val_loss: 1.2353
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.5378 - loss: 1.1420 - val_accuracy: 0.3958 - val_loss: 1.3751
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.5430 - loss: 1.1059 - val_accuracy: 0.

In [78]:
loss, acc = model.evaluate(X_test, y_test, verbose=1)

print("4 USER DELTA CNN ACCURACY:", acc)

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.4375 - loss: 3.5988
4 USER DELTA CNN ACCURACY: 0.4375


In [79]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

print(confusion_matrix(y_true, y_pred))

print(
    classification_report(
        y_true,
        y_pred,
        target_names=["S01", "S02", "S03", "S04"]
    )
)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
[[32 17  0 11]
 [ 4 30  9 17]
 [ 9 19 14 18]
 [ 6 20  5 29]]
              precision    recall  f1-score   support

         S01       0.63      0.53      0.58        60
         S02       0.35      0.50      0.41        60
         S03       0.50      0.23      0.32        60
         S04       0.39      0.48      0.43        60

    accuracy                           0.44       240
   macro avg       0.47      0.44      0.43       240
weighted avg       0.47      0.44      0.43       240

